# Symulator tomografu - GUI w notatniku

Ten notatnik uruchamia interfejs oparty o `ipywidgets`.

## Co potrafi GUI
- wybór obrazu `.jpg` z katalogu `tomograf-obrazy`
- konfiguracja parametrów rekonstrukcji
- podgląd: obraz oryginalny, sinogram i rekonstrukcja
- wykres historii MSE
- zapis wyniku do DICOM
- batch dla wszystkich obrazów `.jpg`

Uruchom komórki od góry do dołu i na końcu klikaj przyciski w panelu.

## 1) Importy i konfiguracja
Poniższa komórka ładuje moduły projektu i przygotowuje foldery wejścia/wyjścia.

In [1]:
# Importy i konfiguracja

from pathlib import Path
import datetime
import time

import imageio.v2 as iio
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from skimage.transform import resize

import filter
import save_dicom
import utils

BASE_DIR = Path('.').resolve()
IMAGES_DIR = BASE_DIR / 'tomograf-obrazy'
DICOM_DIR = BASE_DIR / 'dicom'
DICOM_DIR.mkdir(parents=True, exist_ok=True)

APP_STATE = {
    'last_original': None,
    'last_sinogram': None,
    'last_raw_reconstruction': None,
    'last_filtered_reconstruction': None,
    'last_mse_raw': None,
    'last_mse_filtered': None,
    'last_filename': None,
    'last_params': None,
    'iterative_sinogram': None,
    'iterative_snapshots': None,
    'iterative_mse': None,
    'last_loaded_dicom': None,
}

print(f'Katalog obrazow: {IMAGES_DIR}')
print(f'Katalog DICOM:   {DICOM_DIR}')

Katalog obrazow: /Users/krzysztofzaporowski/Desktop/Studia/InformatykaWMedycynie/tomograf/tomograf-obrazy
Katalog DICOM:   /Users/krzysztofzaporowski/Desktop/Studia/InformatykaWMedycynie/tomograf/dicom


In [2]:
# Helpery backendowe wykorzystywane przez GUI

def available_image_files():
    if not IMAGES_DIR.exists():
        return []
    return sorted([p.name for p in IMAGES_DIR.glob('*.jpg')])


def available_dicom_files():
    if not DICOM_DIR.exists():
        return []
    return sorted([p.name for p in DICOM_DIR.glob('*.dcm')])


def scans_from_delta(delta_alpha_deg):
    if delta_alpha_deg <= 0:
        raise ValueError('Delta alfa musi byc dodatnia.')
    return max(1, int(round(360.0 / float(delta_alpha_deg))))


def load_grayscale_image(filename, image_size):
    path = IMAGES_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Nie znaleziono pliku: {path}')

    image = iio.imread(path.as_posix(), mode='L')
    image = resize(image, (image_size, image_size), preserve_range=True, anti_aliasing=True)
    return image.astype(np.float64)


def scanner_preview(detectors, spread, image_size=100, alpha_deg=0):
    center = image_size / 2
    radius = image_size * 0.7

    emitter, detector_points = utils.wyznacz_pozycje_czujnikow(
        alpha_deg,
        detectors,
        spread,
        radius,
        center,
        center,
    )

    plt.figure(figsize=(6, 6))
    plt.plot(center, center, 'k+', markersize=15, label='Srodek obrotu')
    plt.plot(emitter[0], emitter[1], 'ro', markersize=10, label='Emiter (E)')

    for det in detector_points:
        plt.plot(det[0], det[1], 'bo', markersize=5)
        plt.plot([emitter[0], det[0]], [emitter[1], det[1]], 'gray', alpha=0.3, linestyle='--')

    plt.xlim(-20, image_size + 20)
    plt.ylim(-20, image_size + 20)
    plt.title(f'Uklad tomografu: Alfa={alpha_deg} stopni, n={detectors}, rozpietosc={spread} stopni')
    plt.legend()
    plt.grid(True)
    plt.show()


def compute_sinogram_and_reconstruction(filename, image_size, detectors, delta_alpha, spread):
    scans = scans_from_delta(delta_alpha)
    image = load_grayscale_image(filename, image_size)
    wymiar_y, wymiar_x = image.shape

    sinogram = utils.stworz_sinogram(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        image,
    )

    reconstruction, mse_history = utils.rekonstrukcja_obrazu(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        sinogram,
        image,
    )

    return image, scans, sinogram, reconstruction, mse_history


def compute_iterative_reconstruction(filename, image_size, detectors, delta_alpha, spread):
    scans = scans_from_delta(delta_alpha)
    image = load_grayscale_image(filename, image_size)
    wymiar_y, wymiar_x = image.shape
    sinogram, snapshots, mse_history = utils.rekonstrukcja_iteracyjna(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        image,
    )
    return image, scans, sinogram, snapshots, mse_history


def compute_filtered_reconstruction(image, sinogram, detectors, scans, spread, filter_method='fft'):
    wymiar_y, wymiar_x = image.shape
    sinogram_filtrowany = filter.filtruj_sinogram(sinogram, metoda=filter_method)

    rekonstrukcja_filtrowana, mse_historia_filtrowana = utils.rekonstrukcja_obrazu(
        wymiar_x,
        wymiar_y,
        detectors,
        scans,
        spread,
        sinogram_filtrowany,
        image,
    )

    return sinogram_filtrowany, rekonstrukcja_filtrowana, mse_historia_filtrowana


def plot_single_run(image, sinogram, reconstruction):
    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(image, cmap='gray')
    ax1.set_title('Oryginalny obraz')

    ax2.imshow(sinogram, cmap='gray', aspect='auto')
    ax2.set_title('Wygenerowany Sinogram')
    ax2.set_xlabel('Kat obrotu (skan)')
    ax2.set_ylabel('Numer detektora')
    plt.show()

    plt.imshow(reconstruction, cmap='gray')
    plt.title('Rekonstruowany obraz')
    plt.axis('off')
    plt.show()


def build_iterative_view(image, sinogram, snapshots, mse_history):
    if not snapshots:
        raise RuntimeError('Brak migawek iteracyjnych do wyswietlenia.')

    slider = widgets.IntSlider(
        value=1,
        min=1,
        max=len(snapshots),
        step=1,
        description='Krok:',
        continuous_update=False,
    )
    out = widgets.Output()

    def render(step):
        with out:
            clear_output(wait=True)
            idx = step - 1
            _, axes = plt.subplots(1, 3, figsize=(18, 5))

            axes[0].imshow(image, cmap='gray')
            axes[0].set_title('Oryginalny obraz')
            axes[0].axis('off')

            partial = np.zeros_like(sinogram)
            partial[:, :step] = sinogram[:, :step]
            axes[1].imshow(partial, cmap='gray', aspect='auto')
            axes[1].set_title(f'Sinogram do kroku {step}')
            axes[1].set_xlabel('Skan')
            axes[1].set_ylabel('Detektor')

            axes[2].imshow(snapshots[idx], cmap='gray')
            axes[2].set_title(f'Rekonstrukcja iteracyjna krok {step}')
            axes[2].axis('off')

            plt.tight_layout()
            plt.show()

            mse = mse_history[idx] if idx < len(mse_history) else float('nan')
            print(f'MSE po kroku {step}: {mse:.6f}')

    slider.observe(lambda change: render(change['new']) if change['name'] == 'value' else None, names='value')
    render(1)
    return widgets.VBox([slider, out])


def plot_filtered_run(image, sinogram_filtrowany, rekonstrukcja_filtrowana):
    _, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    ax1.imshow(sinogram_filtrowany, cmap='gray')
    ax1.set_title('Sinogram po filtracji')

    ax2.imshow(rekonstrukcja_filtrowana, cmap='gray')
    ax2.set_title('Rekonstruowany obraz po filtracji')

    ax3.imshow(image, cmap='gray')
    ax3.set_title('Oryginalny obraz')
    plt.show()


def save_reconstruction_to_dicom(first_name, last_name, pesel, study_date, comment):
    reconstruction = APP_STATE.get('last_filtered_reconstruction')
    if reconstruction is None:
        reconstruction = APP_STATE.get('last_raw_reconstruction')
    if reconstruction is None:
        raise RuntimeError('Najpierw uruchom rekonstrukcje.')

    znaczek_czasu = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    nazwa_pliku = DICOM_DIR / f"rekonstrukcja_{first_name}_{last_name}_{znaczek_czasu}.dcm"

    save_dicom.zapisz_dicom(
        reconstruction,
        nazwa_pliku.as_posix(),
        (first_name, last_name, pesel),
        study_date,
        comment,
    )

    return nazwa_pliku


def show_dicom_preview(filename):
    path = DICOM_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Nie znaleziono pliku DICOM: {path}')

    obraz, meta = save_dicom.odczytaj_dicom(path.as_posix())
    APP_STATE['last_loaded_dicom'] = {'filename': filename, 'meta': meta}

    plt.figure(figsize=(5, 5))
    plt.imshow(obraz, cmap='gray')
    plt.title(f"DICOM: {filename}")
    plt.axis('off')
    plt.show()

    print('Metadane DICOM:')
    print(f" - PatientName: {meta['PatientName']}")
    print(f" - PatientID: {meta['PatientID']}")
    print(f" - StudyDate: {meta['StudyDate']}")
    print(f" - StudyDescription: {meta['StudyDescription']}")


def run_batch_full(image_size, detectors, delta_alpha, spread, filter_method='fft'):
    scans = scans_from_delta(delta_alpha)
    files = available_image_files()
    if not files:
        raise RuntimeError('Brak plikow .jpg w katalogu tomograf-obrazy.')

    summary = []
    for file_name in files:
        image, _, sinogram, _, mse_historia = compute_sinogram_and_reconstruction(
            file_name,
            image_size,
            detectors,
            delta_alpha,
            spread,
        )

        _, _, mse_historia_filtrowana = compute_filtered_reconstruction(
            image,
            sinogram,
            detectors,
            scans,
            spread,
            filter_method=filter_method,
        )

        mse_raw = mse_historia[-1] if mse_historia else float('nan')
        mse_filt = mse_historia_filtrowana[-1] if mse_historia_filtrowana else float('nan')
        summary.append((file_name, mse_raw, mse_filt))

    return summary


def analyze_sampling_quality(filename, image_size, base_detectors, base_delta_alpha, base_spread, filter_method='convolve'):
    image = load_grayscale_image(filename, image_size)

    def run_case(detectors, delta_alpha, spread):
        scans = scans_from_delta(delta_alpha)
        sinogram = utils.stworz_sinogram(
            image.shape[1],
            image.shape[0],
            int(detectors),
            scans,
            int(spread),
            image,
        )
        _, mse_hist = utils.rekonstrukcja_obrazu(
            image.shape[1],
            image.shape[0],
            int(detectors),
            scans,
            int(spread),
            sinogram,
            image,
        )
        _, _, mse_hist_f = compute_filtered_reconstruction(
            image,
            sinogram,
            int(detectors),
            scans,
            int(spread),
            filter_method=filter_method,
        )
        return mse_hist[-1] if mse_hist else float('nan'), mse_hist_f[-1] if mse_hist_f else float('nan')

    delta_values = sorted(set([max(1.0, base_delta_alpha / 2), float(base_delta_alpha), min(30.0, base_delta_alpha * 2)]))
    detector_values = sorted(set([max(30, base_detectors // 2), int(base_detectors), min(360, base_detectors * 2)]))
    spread_values = sorted(set([max(30, base_spread // 2), int(base_spread), min(270, base_spread + 60)]))

    result = {'delta_alpha': [], 'detectors': [], 'spread': []}

    for v in delta_values:
        mse_raw, mse_filt = run_case(base_detectors, v, base_spread)
        result['delta_alpha'].append((v, mse_raw, mse_filt))

    for v in detector_values:
        mse_raw, mse_filt = run_case(v, base_delta_alpha, base_spread)
        result['detectors'].append((v, mse_raw, mse_filt))

    for v in spread_values:
        mse_raw, mse_filt = run_case(base_detectors, base_delta_alpha, v)
        result['spread'].append((v, mse_raw, mse_filt))

    return result


def plot_sampling_analysis(results):
    _, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, key, label in [
        (axes[0], 'delta_alpha', 'Delta alfa [stopnie]'),
        (axes[1], 'detectors', 'Liczba detektorow'),
        (axes[2], 'spread', 'Rozpietosc [stopnie]'),
    ]:
        values = [row[0] for row in results[key]]
        mse_raw = [row[1] for row in results[key]]
        mse_filt = [row[2] for row in results[key]]
        ax.plot(values, mse_raw, marker='o', label='MSE bez filtra')
        ax.plot(values, mse_filt, marker='s', label='MSE z filtrem')
        ax.set_xlabel(label)
        ax.set_ylabel('MSE')
        ax.grid(True)
        ax.legend()

    plt.tight_layout()
    plt.show()

## 2) Panel GUI
Poniższa komórka buduje interfejs i podłącza akcje:
- `Uruchom rekonstrukcję`
- `Zapisz aktualny wynik do DICOM`
- `Batch dla wszystkich obrazów`

In [9]:
def build_gui():
    image_files = available_image_files()
    dicom_files = available_dicom_files()

    image_dropdown = widgets.Dropdown(
        options=image_files,
        description='Plik:',
        value=image_files[0] if image_files else None,
        layout=widgets.Layout(width='430px'),
    )
    image_size_slider = widgets.IntSlider(value=100, min=64, max=256, step=4, description='Rozmiar:')
    detectors_slider = widgets.IntSlider(value=180, min=30, max=360, step=10, description='Detektory:')
    delta_alpha_slider = widgets.FloatSlider(value=2.0, min=1.0, max=12.0, step=0.5, description='Delta alfa:')
    spread_slider = widgets.IntSlider(value=180, min=30, max=270, step=5, description='Rozpietosc:')
    use_filter_checkbox = widgets.Checkbox(value=True, description='Pokaz filtracje')
    filter_method_dropdown = widgets.Dropdown(
        options=[('Splot (Ram-Lak)', 'convolve'), ('Transformata Fouriera', 'fft')],
        value='convolve',
        description='Filtr:',
    )

    button_width = '250px'
    preview_button = widgets.Button(description='Podglad ukladu czujnikow', layout=widgets.Layout(width=button_width))
    run_button = widgets.Button(description='Uruchom rekonstrukcje', button_style='primary', layout=widgets.Layout(width=button_width))
    iterative_button = widgets.Button(description='Tryb iteracyjny (suwak)', button_style='warning', layout=widgets.Layout(width=button_width))
    analyze_button = widgets.Button(description='Analiza MSE parametrow', button_style='info', layout=widgets.Layout(width=button_width))
    batch_button = widgets.Button(description='Batch dla wszystkich obrazow', layout=widgets.Layout(width=button_width))
    dicom_save_button = widgets.Button(description='Zapisz DICOM', button_style='success', layout=widgets.Layout(width=button_width))
    dicom_load_button = widgets.Button(description='Wczytaj DICOM', layout=widgets.Layout(width=button_width))

    patient_first = widgets.Text(value='Jan', description='Imie:')
    patient_last = widgets.Text(value='Kowalski', description='Nazwisko:')
    patient_pesel = widgets.Text(value='98051599415', description='PESEL:')
    study_date = widgets.Text(value=datetime.datetime.now().strftime('%Y-%m-%d'), description='Data bad.:')
    patient_comment = widgets.Text(value='Rekonstrukcja tomograficzna', description='Komentarz:')

    dicom_dropdown = widgets.Dropdown(
        options=dicom_files,
        description='DICOM:',
        value=dicom_files[0] if dicom_files else None,
        layout=widgets.Layout(width='430px'),
    )

    status_html = widgets.HTML(value='<b>Status:</b> gotowy')
    output = widgets.Output()

    def update_status(message, level='info'):
        colors = {'info': '#1f6feb', 'ok': '#1a7f37', 'warn': '#9a6700', 'error': '#cf222e'}
        status_html.value = f"<b>Status:</b> <span style='color:{colors.get(level, '#1f6feb')}'>{message}</span>"

    def refresh_dicom_dropdown(select_latest=False):
        files = available_dicom_files()
        dicom_dropdown.options = files
        if not files:
            dicom_dropdown.value = None
            return
        if select_latest:
            dicom_dropdown.value = files[-1]
        elif dicom_dropdown.value not in files:
            dicom_dropdown.value = files[0]

    def on_preview_clicked(_):
        with output:
            clear_output(wait=True)
            scanner_preview(
                detectors=detectors_slider.value,
                spread=spread_slider.value,
                image_size=image_size_slider.value,
                alpha_deg=0,
            )
        update_status('Wyswietlono podglad ukladu czujnikow.', 'ok')

    def on_run_clicked(_):
        with output:
            clear_output(wait=True)
            if not image_dropdown.value:
                update_status('Brak plikow .jpg do przetworzenia.', 'error')
                return

            start = time.perf_counter()
            try:
                image, scans, sinogram, rekonstrukcja, mse_historia = compute_sinogram_and_reconstruction(
                    filename=image_dropdown.value,
                    image_size=image_size_slider.value,
                    detectors=detectors_slider.value,
                    delta_alpha=delta_alpha_slider.value,
                    spread=spread_slider.value,
                )
                plot_single_run(image, sinogram, rekonstrukcja)
                utils.pokaz_historię_mse(mse_historia, tytul='MSE podczas iteracji rekonstrukcji')

                APP_STATE['last_original'] = image
                APP_STATE['last_sinogram'] = sinogram
                APP_STATE['last_raw_reconstruction'] = rekonstrukcja
                APP_STATE['last_mse_raw'] = mse_historia
                APP_STATE['last_filename'] = image_dropdown.value
                APP_STATE['last_params'] = {
                    'image_size': image_size_slider.value,
                    'detectors': detectors_slider.value,
                    'delta_alpha': delta_alpha_slider.value,
                    'scans': scans,
                    'spread': spread_slider.value,
                    'filter_method': filter_method_dropdown.value,
                }

                if use_filter_checkbox.value:
                    sinogram_filtrowany, rekonstrukcja_filtrowana, mse_historia_filtrowana = compute_filtered_reconstruction(
                        image,
                        sinogram,
                        detectors_slider.value,
                        scans,
                        spread_slider.value,
                        filter_method=filter_method_dropdown.value,
                    )
                    plot_filtered_run(image, sinogram_filtrowany, rekonstrukcja_filtrowana)
                    utils.pokaz_historię_mse(mse_historia_filtrowana, tytul='MSE po wlaczeniu filtracji')
                    APP_STATE['last_filtered_reconstruction'] = rekonstrukcja_filtrowana
                    APP_STATE['last_mse_filtered'] = mse_historia_filtrowana
                else:
                    APP_STATE['last_filtered_reconstruction'] = None
                    APP_STATE['last_mse_filtered'] = None

                elapsed = time.perf_counter() - start
                update_status(
                    f"Gotowe w {elapsed:.2f} s. Skanow: {scans}. Filtr: {filter_method_dropdown.value}",
                    'ok',
                )
            except Exception as exc:
                update_status(f'Blad: {exc}', 'error')

    def on_iterative_clicked(_):
        with output:
            clear_output(wait=True)
            if not image_dropdown.value:
                update_status('Brak plikow .jpg do przetworzenia.', 'error')
                return

            start = time.perf_counter()
            try:
                image, scans, sinogram, snapshots, mse_history = compute_iterative_reconstruction(
                    filename=image_dropdown.value,
                    image_size=image_size_slider.value,
                    detectors=detectors_slider.value,
                    delta_alpha=delta_alpha_slider.value,
                    spread=spread_slider.value,
                )

                APP_STATE['iterative_sinogram'] = sinogram
                APP_STATE['iterative_snapshots'] = snapshots
                APP_STATE['iterative_mse'] = mse_history

                display(build_iterative_view(image, sinogram, snapshots, mse_history))

                elapsed = time.perf_counter() - start
                update_status(
                    f"Tryb iteracyjny gotowy ({scans} krokow) w {elapsed:.2f} s.",
                    'ok',
                )
            except Exception as exc:
                update_status(f'Blad trybu iteracyjnego: {exc}', 'error')

    def on_analyze_clicked(_):
        with output:
            clear_output(wait=True)
            if not image_dropdown.value:
                update_status('Brak plikow .jpg do analizy.', 'error')
                return

            start = time.perf_counter()
            try:
                wyniki = analyze_sampling_quality(
                    filename=image_dropdown.value,
                    image_size=image_size_slider.value,
                    base_detectors=detectors_slider.value,
                    base_delta_alpha=delta_alpha_slider.value,
                    base_spread=spread_slider.value,
                    filter_method=filter_method_dropdown.value,
                )
                plot_sampling_analysis(wyniki)

                print('Zmiana MSE przy zmianie parametrow probkowania:')
                for klucz, etykieta in [
                    ('delta_alpha', 'Delta alfa'),
                    ('detectors', 'Liczba detektorow'),
                    ('spread', 'Rozpietosc'),
                ]:
                    print(f'\n{etykieta}:')
                    print('wartosc | MSE bez filtra | MSE z filtrem')
                    for value, mse_raw, mse_filt in wyniki[klucz]:
                        print(f'{value} | {mse_raw:.6f} | {mse_filt:.6f}')

                elapsed = time.perf_counter() - start
                update_status(f'Analiza MSE zakonczona w {elapsed:.2f} s.', 'ok')
            except Exception as exc:
                update_status(f'Analiza nie powiodla sie: {exc}', 'error')

    def on_save_dicom_clicked(_):
        with output:
            try:
                filename = save_reconstruction_to_dicom(
                    patient_first.value.strip(),
                    patient_last.value.strip(),
                    patient_pesel.value.strip(),
                    study_date.value.strip(),
                    patient_comment.value.strip(),
                )
                print(f'Zapisano plik: {filename}')
                refresh_dicom_dropdown(select_latest=True)
                update_status(f'Zapisano DICOM: {filename.name}', 'ok')
            except Exception as exc:
                update_status(f'Nie udalo sie zapisac DICOM: {exc}', 'error')

    def on_load_dicom_clicked(_):
        with output:
            if not dicom_dropdown.value:
                update_status('Brak pliku DICOM do wczytania.', 'warn')
                return
            try:
                clear_output(wait=True)
                show_dicom_preview(dicom_dropdown.value)
                update_status(f'Wczytano DICOM: {dicom_dropdown.value}', 'ok')
            except Exception as exc:
                update_status(f'Nie udalo sie wczytac DICOM: {exc}', 'error')

    def on_batch_clicked(_):
        with output:
            clear_output(wait=True)
            start = time.perf_counter()
            try:
                summary = run_batch_full(
                    image_size=image_size_slider.value,
                    detectors=detectors_slider.value,
                    delta_alpha=delta_alpha_slider.value,
                    spread=spread_slider.value,
                    filter_method=filter_method_dropdown.value,
                )
                print('Podsumowanie batch (plik -> mse_surowe | mse_filtrowane):')
                for file_name, mse_raw, mse_filt in summary:
                    print(f' - {file_name}: {mse_raw:.6f} | {mse_filt:.6f}')
                elapsed = time.perf_counter() - start
                update_status(
                    f"Batch zakonczony w {elapsed:.2f} s. Filtr: {filter_method_dropdown.value}",
                    'ok',
                )
            except Exception as exc:
                update_status(f'Batch przerwany: {exc}', 'error')

    preview_button.on_click(on_preview_clicked)
    run_button.on_click(on_run_clicked)
    iterative_button.on_click(on_iterative_clicked)
    analyze_button.on_click(on_analyze_clicked)
    dicom_save_button.on_click(on_save_dicom_clicked)
    dicom_load_button.on_click(on_load_dicom_clicked)
    batch_button.on_click(on_batch_clicked)

    params_box = widgets.VBox([
        image_dropdown,
        image_size_slider,
        detectors_slider,
        delta_alpha_slider,
        spread_slider,
        use_filter_checkbox,
        filter_method_dropdown,
    ])

    action_box = widgets.HBox([preview_button, run_button, iterative_button])
    analyze_box = widgets.HBox([analyze_button, batch_button])
    patient_box = widgets.VBox([
        widgets.HTML('<b>Dane do zapisu DICOM</b>'),
        patient_first,
        patient_last,
        patient_pesel,
        study_date,
        patient_comment,
        dicom_save_button,
    ])
    dicom_box = widgets.VBox([
        widgets.HTML('<b>Odczyt DICOM</b>'),
        dicom_dropdown,
        dicom_load_button,
    ])

    root = widgets.VBox([params_box, action_box, analyze_box, patient_box, dicom_box, status_html, output])
    return root

In [10]:
# Quick start: uruchom GUI jednym kliknieciem

gui = build_gui()
display(gui)

## Uwagi
Interfejs GUI pokrywa logikę pierwotnego notatnika: podglad ukladu, sinogram, rekonstrukcja, filtracja, zapis DICOM i batch.